
# Card &middot; External data

| | |
|---|---|
| **time** | ~50 minutes &mdash; **timebox this or it eats your afternoon** |
| **GPU** | not needed |
| **typical gain** | large on some endpoints, zero on others |
| **needs** | nothing; runs standalone |

Nearly every top-20 finisher in the real challenge added external public data,
and four of the top five also had proprietary data. If there is a "more is
more" lesson in this challenge, it is here.

But external data is not free, and the cost is not download time. It is that
**a number from someone else's assay is not the same measurement as a number
from yours**, and making them comparable is fiddly, unglamorous, and the place
where most of the mistakes happen.

You will spend most of this notebook on that, not on modelling. That is the
correct allocation.

In [ ]:
# Run me first.
%pip -q install rdkit pandas numpy scipy scikit-learn huggingface_hub fsspec lightgbm matplotlib seaborn PyTDC

# Get common.py. If you uploaded it yourself (folder icon in the left sidebar),
# this leaves your copy alone -- it only downloads when the file is missing.
!test -s common.py || wget -q -O common.py https://raw.githubusercontent.com/CHANGE-ME/admet-hackathon/main/common.py

import os, sys
assert os.path.exists("common.py") and os.path.getsize("common.py") > 1000, (
    "common.py is missing or truncated. Upload it using the folder icon in the "
    "left sidebar, then re-run this cell.")

sys.modules.pop("common", None)   # force a fresh read if you just re-uploaded it
import common
common.setup(pair="CHANGE-ME")

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from lightgbm import LGBMRegressor
sns.set_style("whitegrid")

train = common.load_train()
test  = common.load_test()
fold, _ = common.load_split(train)

---
## 1. What is out there

| source | what it has | notes |
|---|---|---|
| **ASAP Discovery / Polaris** antiviral ADMET | LogD, KSOL, HLM, MLM, permeability | closest in spirit &mdash; also a real campaign, also released unblinded after a challenge |
| **TDC** (Therapeutics Data Commons) | Caco-2, solubility, PPB, clearance | curated for breadth, so chemically very diverse |
| **ChEMBL** | enormous, LogD especially | needs heavy filtering; assay conditions vary wildly |
| **Galapagos**, **Novartis** released sets | multi-endpoint pharma data | both were used by challenge participants |

Loaded here as a pre-built artifact. The download and parsing has been done for
you because it is friction with no lesson in it; the *harmonisation* below has
not, because that is the whole point.

In [ ]:
try:
    ext = common.load_artifact("external_raw.parquet")
except Exception as exc:
    print("artifact unavailable, fetching live:", exc)
    from tdc.single_pred import ADME
    frames = []
    for tdc_name, label in [("Caco2_Wang", "caco2"),
                            ("Solubility_AqSolDB", "solubility"),
                            ("PPBR_AZ", "ppb_human"),
                            ("Clearance_Hepatocyte_AZ", "clearance_human"),
                            ("Lipophilicity_AstraZeneca", "logd")]:
        try:
            d = ADME(name=tdc_name).get_data()
            frames.append(pd.DataFrame({"SMILES": d["Drug"], "value": d["Y"],
                                        "source_endpoint": label,
                                        "source": f"TDC:{tdc_name}"}))
        except Exception as e:
            print("  skipped", tdc_name, e)
    ext = pd.concat(frames, ignore_index=True)

print(len(ext), "external rows")
ext.groupby("source_endpoint")["value"].describe().round(2)

---
## 2. Harmonisation &mdash; the part that actually matters

Below is a mapping from external endpoints onto ours. **It has at least one
error in it.** Not a typo &mdash; a plausible, professional-looking mistake of
exactly the kind that has been published in real ADMET tooling.

Your job is to find it before you train on it.

Three things to check for every row, in this order:

1. **Species.** Is the external assay measuring the same organism as ours?
2. **Scale and units.** Is it already logged? Micromolar or molar? Per mg or
   per kg?
3. **The assay itself.** Intrinsic clearance and plasma clearance are different
   quantities. Kinetic and thermodynamic solubility are different experiments.

In [ ]:
HARMONISATION = {
    # source_endpoint  ->  (our endpoint,        already_log, multiplier)
    "logd":            ("LogD",                  True,  1.0),
    "solubility":      ("LogS",                  True,  1.0),
    "caco2":           ("Log_Caco_Papp_AB",      True,  1.0),
    "ppb_human":       ("Log_Mouse_PPB",         False, 1.0),
    "clearance_human": ("Log_HLM_CLint",         False, 1.0),
}
pd.DataFrame(HARMONISATION, index=["our endpoint", "already log?", "multiplier"]).T

### &#9654;&#65039; Predict first

**Read the table above against the three checks. Which row would you refuse to use, and why?**

*One of them is wrong for a reason no unit conversion can fix.*

Write your answer here before running the next cell &mdash; one line is enough:

> `your prediction:`

### Verify by distribution, never by documentation

The reliable way to catch these is to plot the external distribution against
ours. If the shapes do not line up, something is wrong regardless of what the
documentation claims.

This is not a hypothetical precaution. When OpenADMET benchmarked ADMET-AI,
its own documentation stated Caco-2 permeability was in `log(10^-6 cm/s)`.
Plotting the distribution showed it was actually in `log(cm/s)` &mdash; a factor
of a million, silently, in a well-maintained and widely used tool.

In [ ]:
fig, axes = plt.subplots(1, len(HARMONISATION), figsize=(4 * len(HARMONISATION), 3.4))
for ax, (src, (ours, is_log, mult)) in zip(np.atleast_1d(axes), HARMONISATION.items()):
    v = ext.loc[ext["source_endpoint"] == src, "value"].dropna()
    if not is_log:
        v = np.log10(v.clip(lower=1e-9) * mult)
    ax.hist(v, bins=40, density=True, alpha=.6, label="external")
    ax.hist(train[ours].dropna(), bins=40, density=True, alpha=.6, label="ExpansionRx")
    ax.set_title(f"{src}\n-> {ours}", fontsize=9)
    ax.legend(fontsize=7); ax.set_yticks([])
plt.tight_layout(); plt.show()

Look for: distributions offset by a constant (a units problem, and
fixable), distributions of completely different width (a scale problem, also
usually fixable), and distributions that simply do not describe the same
quantity (not fixable &mdash; drop the source).

Write your corrected table below. Deleting a row is a legitimate answer and
often the right one.

In [ ]:
MY_HARMONISATION = {
    # copy the rows you trust, fix the ones you can, delete the ones you cannot
}

In [ ]:
#@title Reveal and discussion { display-mode: "form" }
print('''
ppb_human -> Log_Mouse_PPB is the bad one.

TDC's PPBR_AZ measures plasma protein binding in HUMAN plasma. The
ExpansionRx MPPB endpoint is MOUSE. Protein binding varies substantially
between species: albumin and alpha-1-acid glycoprotein differ in sequence,
abundance and binding affinity. A compound 99% bound in human plasma can be
95% bound in mouse -- a 5x difference in free fraction, which is the number
that determines whether the drug works.

No unit conversion repairs this. It is a different measurement.

When OpenADMET ran their own zero-shot benchmark they hit exactly this and
dropped PPB from the web-tool comparison entirely, because both ADMET-AI and
ADMETlab 3.0 report human PPB while the challenge data is mouse.

clearance_human -> Log_HLM_CLint deserves suspicion too. TDC's
Clearance_Hepatocyte_AZ is hepatocyte clearance, not microsomal; different
system, different scaling. You *can* relate them, but you should know you are
making an assumption. ADMETlab 3.0 reports PLASMA clearance, which needs the
Well-Stirred model to invert -- and that inversion amplifies error badly as
clearance approaches hepatic blood flow. Usable for a retrospective analysis;
not something to build a prospective model on.
''')

---
## 3. Will it help? Check chemical space first

Before training on anything, ask whether the external molecules are anywhere
near yours. From `01_eda`: below a Tanimoto of about 0.27, a nearest neighbour
is indistinguishable from a random molecule, and adds no usable information.

### &#9654;&#65039; Predict first

**Rank the endpoints by how much you expect external data to help. Commit to an order before you compute anything.**

*Which endpoints have abundant, consistently-measured public data? Which depend on a specific biology that public assays measure differently?*

Write your answer here before running the next cell &mdash; one line is enough:

> `your prediction:`

In [ ]:
fp_train = common.morgan_fingerprints(train[common.SMILES_COL])
rows = []
for src in ext["source_endpoint"].unique():
    smi = ext.loc[ext["source_endpoint"] == src, "SMILES"].dropna()
    smi = smi.sample(min(1500, len(smi)), random_state=0)
    fp_ext = common.morgan_fingerprints(smi)
    nn = common.nearest_neighbour_similarity(fp_train, fp_ext)
    rows.append({"source": src, "median NN sim to ExpansionRx": np.nanmedian(nn),
                 "% below noise floor":
                     100 * np.nanmean(nn < common.MORGAN2_NOISE_FLOOR)})
overlap = pd.DataFrame(rows).round(3)
overlap

Compare this against your ranking. In the real challenge, LogD
was the one endpoint where models trained purely on public data came close to
models trained on the project's own data &mdash; because LogD is additive,
publicly abundant, and the chemical space overlap is good enough. Clearance and
permeability were the opposite: public models produced negative R&sup2;.

---
## 4. Train with the augmented data

In [ ]:
def augmented_training_set(our_endpoint, harmonisation):
    """Stack ExpansionRx rows with harmonised external rows for one endpoint."""
    tr = train[(fold == "train").to_numpy()]
    base = pd.DataFrame({common.SMILES_COL: tr[common.SMILES_COL],
                         "y": tr[our_endpoint], "origin": "expansionrx"}).dropna()
    add = []
    for src, (ours, is_log, mult) in harmonisation.items():
        if ours != our_endpoint:
            continue
        e = ext[ext["source_endpoint"] == src].dropna(subset=["value"])
        v = e["value"].astype(float)
        if not is_log:
            v = np.log10(v.clip(lower=1e-9) * mult)
        add.append(pd.DataFrame({common.SMILES_COL: e["SMILES"], "y": v,
                                 "origin": src}))
    return pd.concat([base] + add, ignore_index=True) if add else base


def score_with(endpoint, harmonisation, label):
    data = augmented_training_set(endpoint, harmonisation)
    X = common.rdkit_descriptors(data[common.SMILES_COL])
    va = train[(fold == "val").to_numpy()].reset_index(drop=True)
    Xva = common.rdkit_descriptors(va[common.SMILES_COL])
    A, B = common.clean_features(X, Xva)
    m = LGBMRegressor(n_estimators=500, learning_rate=0.05, verbose=-1, n_jobs=-1)
    m.fit(A, data["y"])
    p = pd.DataFrame({common.ID_COL: va[common.ID_COL], endpoint: m.predict(B)})
    r = common.evaluate(va, p, [endpoint]).loc[endpoint, "RAE"]
    print(f"{endpoint:20s} {label:18s} RAE = {r:.3f}  (n={len(data):,})")
    return r

In [ ]:
for e in ["LogD", "LogS", "Log_Caco_Papp_AB", "Log_HLM_CLint"]:
    score_with(e, {}, "ExpansionRx only")
    score_with(e, MY_HARMONISATION, "+ external")
    print()

**Report the endpoints where it hurt as well as the ones where it
helped.** A source that damages clearance while improving LogD is the normal
outcome, and "we used external data for LogD only, and here is why" is a better
slide than "we added everything".

### If you have time: weight the external rows down

External measurements are noisier and come from different assays, so treating
them as equal to your project data is optimistic. `LGBMRegressor.fit` takes
`sample_weight` &mdash; try giving external rows a weight of 0.3 and see whether
you keep the gain while losing less on the endpoints it hurt.

---
## Save your work

Give it a name you will recognise at 4pm. `card_ensembles` can combine this
with anything else you have made today.

In [ ]:
X_tr = common.rdkit_descriptors(train[common.SMILES_COL])
X_te = common.rdkit_descriptors(test[common.SMILES_COL])

pred = common.blank_predictions(test)
for e in common.ENDPOINTS:
    data = augmented_training_set(e, MY_HARMONISATION)
    Xa = common.rdkit_descriptors(data[common.SMILES_COL])
    A, B = common.clean_features(Xa, X_te)
    m = LGBMRegressor(n_estimators=500, learning_rate=0.05, verbose=-1, n_jobs=-1)
    m.fit(A, data["y"])
    pred[e] = m.predict(B)

common.save_predictions(pred, "lgbm-external",
                        note="LightGBM, training set augmented with public data")